In [2]:
import arviz as az
from utils import get_data, build_model, plot_ppc
import numpy as np
import seaborn as sns
from bauer.utils.bayes import softplus
import os
import os.path as op
import pandas as pd

bids_folder='/Users/mrenke/data/ds-stressrisk'
model_label = '1'


/Users/mrenke/mambaforge/envs/behav_fit/lib/python3.10/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '
/Users/mrenke/mambaforge/envs/behav_fit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = get_data(bids_folder)
model = build_model(model_label, df)
model.build_estimation_model()

idata = az.from_netcdf(op.join(bids_folder, f'derivatives/cogmodels/model-{model_label}_trace.netcdf'))


In [4]:
#from utils_02 import get_subwise_params
def get_subwise_params(idata, param_name):
    df_param= idata.posterior[param_name].to_dataframe()    
    df_param.columns.name = 'parameter'
    df_param.index = df_param.index.set_names(['chain','draw','subject','regressor']) 
    df_param = df_param.stack().to_frame('value')

    df_param = df_param.xs('Intercept', 0,'regressor')
    df_param = df_param.groupby(['subject'])[['value']].mean()
    df_param = df_param.rename(mapper={'value':param_name},axis=1)

    return df_param

In [12]:
params = ['risky_prior_mu','safe_prior_mu','risky_prior_std','safe_prior_std','n1_evidence_sd','n2_evidence_sd']
df_params = pd.DataFrame(index=df.index.get_level_values('subject').unique())
for params in params:
    df_param = get_subwise_params(idata,param_name=params)
    df_params = df_params.merge(df_param, left_on='subject', right_index=True)

df_params.head()

,risky_prior_mu,safe_prior_mu,risky_prior_std,safe_prior_std,n1_evidence_sd,n2_evidence_sd
subject,,,,,,
1,3.511950,2.561211,-0.910787,-1.084006,-1.153115,-1.880260
2,3.369483,3.488596,-0.609618,-1.200628,-1.185866,-1.512110
3,3.194726,2.879364,-1.220158,-0.733372,-1.045374,-1.555680
4,3.202980,3.810281,-1.293644,-0.867947,-1.129907,-1.630347
5,3.497174,3.118516,-0.699055,-1.080456,-1.135886,-1.754081


In [119]:
df

run        rt    n1    n2  prob1  prob2  choice  \
subject trial_nr session                                                    
1       1        1          1  0.476974  13.0  10.0   0.55    1.0    True   
        2        1          1  0.505059   8.0   7.0   0.55    1.0    True   
        3        1          1  0.609260  13.0   7.0   0.55    1.0   False   
        4        1          1  0.522144  38.0  28.0   0.55    1.0    True   
        5        1          1  0.608297  16.0  10.0   0.55    1.0   False   
...                       ...       ...   ...   ...    ...    ...     ...   
61      116      2          6  1.189453  39.0  14.0   0.55    1.0   False   
        117      2          6  0.755411  43.0  20.0   0.55    1.0   False   
        118      2          6  0.905503  34.0  14.0   0.55    1.0    True   
        119      2          6  1.873285  34.0  20.0   0.55    1.0    True   
        120      2          6  0.989251  42.0  28.0   0.55    1.0    True   

                          risky_first  chose_risky  n_risky  n_safe      frac  \
subject trial_nr session                                                        
1       1        1               True        False     13.0    10.0  1.300000   
        2        1               True        False      8.0     7.0  1.142857   
        3        1               True         True     13.0     7.0  1.857143   
        4        1               True        False     38.0    28.0  1.357143   
        5        1               True         True     16.0    10.0  1.600000   
...                               ...          ...      ...     ...       ...   
61      116      2               True         True     39.0    14.0  2.785714   
        117      2               True         True     43.0    20.0  2.150000   
        118      2               True        False     34.0    14.0  2.428571   
        119      2               True        False     34.0    20.0  1.700000   
        120      2               True        False     42.0    28.0  1.500000   

                          log(risky/safe)   log(n1) bin(risky/safe)    p1  \
subject trial_nr session                                                    
1       1        1               0.262364  2.564949             32%  0.55   
        2        1               0.133531  2.079442             20%  0.55   
        3        1               0.619039  2.564949             56%  0.55   
        4        1               0.305382  3.637586             32%  0.55   
        5        1               0.470004  2.772589             44%  0.55   
...                                   ...       ...             ...   ...   
61      116      2               1.024504  3.663562             80%  0.55   
        117      2               0.765468  3.761200             56%  0.55   
        118      2               0.887303  3.526361             68%  0.55   
        119      2               0.530628  3.526361             32%  0.55   
        120      2               0.405465  3.737670             20%  0.55   

                           p2  group  
subject trial_nr session              
1       1        1        1.0    0.0  
        2        1        1.0    0.0  
        3        1        1.0    0.0  
        4        1        1.0    0.0  
        5        1        1.0    0.0  
...                       ...    ...  
61      116      2        1.0    0.0  
        117      2        1.0    0.0  
        118      2        1.0    0.0  
        119      2        1.0    0.0  
        120      2        1.0    0.0  

[11891 rows x 18 columns]

In [25]:
#import pytensor.tensor as np
import numpy as np
import scipy.special as sp

def get_posterior(mu1, sd1, mu2, sd2):
    var1, var2 = sd1**2, sd2**2
    return mu1 + (var1/(var1+var2))*(mu2 - mu1), np.sqrt((var1*var2)/(var1+var2))
def get_diff_dist(mu1, sd1, mu2, sd2):
    return mu2 - mu1, np.sqrt(sd1**2+sd2**2)
def cumulative_normal(x, mu, sd, s=np.sqrt(2.)):
#     Cumulative distribution function for the standard normal distribution
    return np.clip(0.5 + 0.5 *
                   sp.erf((x - mu) / (sd*s)), 1e-9, 1-1e-9)

In [89]:
# ultimate choice between n1 and n2
def gen_choice_pred(df_trial_row,sub_params):
    # subwise params
    threshold = np.log(df_trial_row['p2']/df_trial_row['p1']) #df_params.loc[sub, 'threshold']
    risky_prior_mu = sub_params['risky_prior_mu']
    safe_prior_mu = sub_params['safe_prior_mu']
    risky_prior_std = sub_params['risky_prior_std'] 
    safe_prior_std = sub_params['safe_prior_std']   
    n1_evidence_sd = sub_params['n1_evidence_sd']
    n2_evidence_sd = sub_params['n2_evidence_sd']   

    # trialwise params
    risky_first = df_trial_row['risky_first'].item()
    n1_evidence_mu = np.log(df_trial_row['n1'])
    n2_evidence_mu = np.log(df_trial_row['n2'])   

    # convert from risky/safe to first/second
    n1_prior_mu = np.where(risky_first, risky_prior_mu, safe_prior_mu)
    n1_prior_std = np.where(risky_first, risky_prior_std, safe_prior_std)
    n2_prior_mu = np.where(risky_first, safe_prior_mu, risky_prior_mu)
    n2_prior_std = np.where(risky_first, safe_prior_std, risky_prior_std)

    post_n1_mu, post_n1_sd = get_posterior(n1_prior_mu, 
                                                n1_prior_std, 
                                                n1_evidence_mu, 
                                                n1_evidence_sd
                                                )

    post_n2_mu, post_n2_sd = get_posterior(n2_prior_mu,
                                                n2_prior_std,
                                                n2_evidence_mu, 
                                                n2_evidence_sd)

    diff_mu, diff_sd = get_diff_dist(post_n2_mu, post_n2_sd, post_n1_mu, post_n1_sd)

    p = cumulative_normal(threshold, diff_mu, diff_sd) # p chose 2nd
    choice = np.random.binomial(n=1, p=p, size=1)[0] # 1 = chose 2nd

    if choice == 1:
        if risky_first:
            chose_risky = False
        elif not risky_first:
            chose_risky = True
    elif choice == 0:
        if risky_first:
            chose_risky = True
        elif  not risky_first:
            chose_risky = False  

    return chose_risky


In [122]:
# ultimate choice between risky and safe

def gen_choice_pred(df_trial_row,sub_params):
    # subwise params
    threshold = np.log(1/0.55) #df_params.loc[sub, 'threshold']
    risky_prior_mu = sub_params['risky_prior_mu']
    safe_prior_mu = sub_params['safe_prior_mu']
    risky_prior_std = sub_params['risky_prior_std'] 
    safe_prior_std = sub_params['safe_prior_std']   
    n1_evidence_sd = sub_params['n1_evidence_sd']
    n2_evidence_sd = sub_params['n2_evidence_sd']   

    # trialwise params
    risky_first = df_trial_row['risky_first'].item()
    risky_evidence_mu = np.log(df_trial_row['n_risky'])
    safe_evidence_mu = np.log(df_trial_row['n_safe'])   

    # convert from risky/safe to first/second
    risky_evidence_sd = np.where(risky_first, n1_evidence_sd, n2_evidence_sd)
    safe_evidence_sd = np.where(risky_first, n2_evidence_sd, n1_evidence_sd)

    # inference process
    post_risky_mu, post_risky_sd = get_posterior(risky_prior_mu,
                                                 risky_prior_std,
                                                 risky_evidence_mu,
                                                 risky_evidence_sd
                                                 )

    post_safe_mu, post_safe_sd = get_posterior(safe_prior_mu,
                                               safe_prior_std,
                                               safe_evidence_mu,
                                               safe_evidence_sd)

    diff_mu, diff_sd = get_diff_dist(post_safe_mu, post_safe_sd, post_risky_mu, post_risky_sd)

    p = cumulative_normal(threshold, diff_mu, diff_sd) # p chose 2nd
    choice = np.random.binomial(n=1, p=p, size=1)[0] # 1 = chose 2nd

    chose_risky = bool(choice)

    return chose_risky


In [103]:
#df.set_index('session',append=True, inplace=True)
df.reset_index('run',inplace=True)
df.head()

run        rt    n1    n2  prob1  prob2  choice  \
subject trial_nr session                                                    
1       1        1          1  0.476974  13.0  10.0   0.55    1.0    True   
        2        1          1  0.505059   8.0   7.0   0.55    1.0    True   
        3        1          1  0.609260  13.0   7.0   0.55    1.0   False   
        4        1          1  0.522144  38.0  28.0   0.55    1.0    True   
        5        1          1  0.608297  16.0  10.0   0.55    1.0   False   

                          risky_first  chose_risky  n_risky  n_safe      frac  \
subject trial_nr session                                                        
1       1        1               True        False     13.0    10.0  1.300000   
        2        1               True        False      8.0     7.0  1.142857   
        3        1               True         True     13.0     7.0  1.857143   
        4        1               True        False     38.0    28.0  1.357143   
        5        1               True         True     16.0    10.0  1.600000   

                          log(risky/safe)   log(n1) bin(risky/safe)    p1  \
subject trial_nr session                                                    
1       1        1               0.262364  2.564949             32%  0.55   
        2        1               0.133531  2.079442             20%  0.55   
        3        1               0.619039  2.564949             56%  0.55   
        4        1               0.305382  3.637586             32%  0.55   
        5        1               0.470004  2.772589             44%  0.55   

                           p2  group  
subject trial_nr session              
1       1        1        1.0    0.0  
        2        1        1.0    0.0  
        3        1        1.0    0.0  
        4        1        1.0    0.0  
        5        1        1.0    0.0

In [132]:
#df.set_index('session',append=True, inplace=True)

subjects = df.index.get_level_values('subject').unique()
session = df.index.get_level_values('session').unique()

choices_pred = pd.DataFrame(index=df.index,columns=['chose_risky_pred'])
for sub in subjects:
    for ses in session:
        df_sub_ses = df.xs((sub,ses), level=('subject', 'session'))
        trials = df_sub_ses.index.get_level_values('trial_nr').unique()
        for trial in trials:
            df_trial_row = df.xs((sub,ses, trial), level=('subject', 'session','trial_nr'))
            sub_params = df_params.loc[sub]
            chose_risky = gen_choice_pred(df_trial_row,sub_params) # ['chose_risky_pred']
            choices_pred.loc[(sub, trial, ses)] = chose_risky




In [133]:
df_comb = df.join(choices_pred)

In [134]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df_comb['chose_risky'], df_comb['chose_risky_pred'])

chi2, p, dof, expected = chi2_contingency(contingency_table)

print(f"chi2: {chi2}, p-value: {p}")


chi2: 0.20792652150657165, p-value: 0.6483974519520211


In [135]:
contingency_table

chose_risky_pred,False,True
chose_risky,,
False,2337,3437
True,2502,3615
